*© 2026 Zettnq · oi-volume-signal-research · MIT License*
### 03. Forward Returns & Raw Baseline 
Calculation of forward returns and direction-adjusted PnL (entry open[t+1]), availability report and raw baseline statistics (existence check for S5). Artifact: ..._forward_returns.parquet.

In [1]:
from pathlib import Path

import oivdc_research  # installed via `pip install -e .`

# repo root: .../repo/src/oivdcr_research/__init__.py -> parents[2] = repo
PROJECT_ROOT = Path(oivdc_research.__file__).resolve().parents[2]

import numpy as np
import pandas as pd

from oivdc_research.config import get_config

# Reproducibility parameters — edit here.
ASSET = "btcusdt"
TIMEFRAME = "1h"

CONFIG = get_config(
    project_root=PROJECT_ROOT,
    raw_csv_path=PROJECT_ROOT / "data" / "raw" / f"{ASSET}_{TIMEFRAME}.csv",
    timeframe=TIMEFRAME,
    asset=ASSET,
)

In [2]:
from oivdc_research.signals import load_signals
from oivdc_research.forward_returns import (
    add_forward_return_columns,
    summarize_forward_return_availability,
    save_forward_returns,
)

df = load_signals(CONFIG)
df = add_forward_return_columns(df, CONFIG)

print(summarize_forward_return_availability(df, CONFIG))
path_fr = save_forward_returns(df, CONFIG)
print(f"Saved: {path_fr.relative_to(CONFIG.project_root).as_posix()}")

   horizon  total_bars  valid_all  missing_all  n_signals  valid_signals  \
0        1       44502      44501            1      16673          16673   
1        2       44502      44500            2      16673          16673   
2        3       44502      44499            3      16673          16673   
3        4       44502      44498            4      16673          16672   
4        5       44502      44497            5      16673          16672   
5       10       44502      44492           10      16673          16671   

   missing_signals  
0                0  
1                0  
2                0  
3                1  
4                1  
5                2  
Saved: data/processed/btcusdt_1h_forward_returns.parquet


In [3]:
from oivdc_research.event_selection import build_events_raw
from oivdc_research.statistics import compute_statistics_for_events_wide, format_stats_table

events_raw = build_events_raw(df, CONFIG)
raw_stats = compute_statistics_for_events_wide(
    events_raw, CONFIG, method_name="raw_signals", sample_type="overlapping"
)
print(format_stats_table(raw_stats))

         method  sample_type signal_scope  horizon  n_events   winrate  \
0   raw_signals  overlapping     combined        1     16673  0.534517   
1   raw_signals  overlapping     combined        2     16673  0.529899   
2   raw_signals  overlapping     combined        3     16673  0.533857   
3   raw_signals  overlapping     combined        4     16672  0.525492   
4   raw_signals  overlapping     combined        5     16672  0.530470   
5   raw_signals  overlapping     combined       10     16671  0.512807   
6   raw_signals  overlapping         long        1      8263  0.534310   
7   raw_signals  overlapping         long        2      8263  0.539635   
8   raw_signals  overlapping         long        3      8263  0.544233   
9   raw_signals  overlapping         long        4      8263  0.532978   
10  raw_signals  overlapping         long        5      8263  0.540240   
11  raw_signals  overlapping         long       10      8262  0.517308   
12  raw_signals  overlapping        sh

The raw signals provide a faint indication (~53% WR) — this is the baseline to which we will return in 09 to show where the edge ‘goes’ when aggregated correctly.